### Transformada Discreta de Fourier (DFT)

A Transformada de Fourier de Tempo Discreto (TFTD) analisa um sinal $x(n)$ no domínio da frequência:

$$X(e^{j\omega}) = \sum_{n=-\infty}^{\infty} x(n)\, e^{-j\omega n} \quad \text{(Eq. de Análise)}$$

$$x(n) = \frac{1}{2\pi} \int_{2\pi} X(e^{j\omega})\, e^{j\omega n}\, d\omega \quad \text{(Eq. de Síntese)}$$

Mas temos dificuldade para representar $X(e^{j\omega})$ em um computador digital. A solução é realizar amostragem no domínio da frequência via **DFT**


### 1. DFT a partir da TFTD

Considere $x(n)$ uma sequência de duração finita de comprimento $L$ (ou seja, $x(n) = 0$ para $n < 0$ e $n \geq L$), assim:

$$X(e^{j\omega}) = \sum_{n=0}^{L-1} x(n)\, e^{-j\omega n}$$

Amostrando $X(e^{j\omega})$ em N frequências igualmente espaçadas:

$$\omega_k = \frac{2\pi k}{N}, \quad k = 0, 1, 2, \ldots, N-1$$

Obtemos a DFT de N pontos:

$$\boxed{X(k) = \sum_{n=0}^{N-1} x(n)\, e^{-j\frac{2\pi k}{N}n}, \quad k = 0, 1, \ldots, N-1}$$

onde $N \geq L$ (condição para evitar time-aliasing).

### 2. Implementação — DFT Direta

Formúla:

$$X(k) = \sum_{n=0}^{N-1} x(n)\, e^{-j\frac{2\pi k}{N}n}$$

In [2]:
# Importando bibliotecas.
import numpy as np
import matplotlib.pyplot as plt
import time

In [3]:
def dft(x, N=None):
    """
    Parâmetros:
        x : array: sinal de entrada (comprimento L)
        N : int: número de pontos da DFT (N >= L), se None, usa N = len(x)
    
    Retorna:
        X : array complexo de N pontos
    """
    
    x = np.array(x, dtype=complex)
    L = len(x)
    
    if N is None:
        N = L
    
    # Verifica condição N >= L (evita time aliasing)
    if N < L:
        raise ValueError(f"N={N} deve ser >= L={L} para evitar time aliasing!")
    
    # Zero-padding: se N > L, completa com zeros até comprimento N
    if N > L:
        x = np.concatenate([x, np.zeros(N - L, dtype=complex)])
        
    X = np.zeros(N, dtype=complex)
    
    for k in range(N):           
        soma = 0 + 0j
        for n in range(N):       
            # Parte exponencial
            exp = -(2 * np.pi * k * n) / N
            soma += x[n] * (np.cos(exp) + 1j * np.sin(exp))
        X[k] = soma
    
    return X

### 3. Custo Computacional da DFT Direta

Para calcular um único valor $X(k)$, o somatório requer:
- N multiplicações complexas
- N-1 adições complexas

$$\text{Multiplicações: } N^2 \qquad \text{Adições: } (N-1)N \longrightarrow
\text{Complexidade: } \mathcal{O}(N^2)$$


In [4]:
def medir_tempo_dft(x, N):
    """
    Recebe um sinal x qualquer e mede o tempo da DFT direta.
    """
    x = np.array(x, dtype=complex)

    # Nossa DFT
    t0 = time.time()
    X_direta = dft(x, N)
    t_direta = (time.time() - t0) * 1000

    print(f"Tamanho do sinal:  N = {N}")
    print(f"DFT Direta:        {t_direta:.4f} ms")

    return X_direta

In [5]:
print("=== Implementação da DFT Direta ===")
print("Digite os valores das amostras separados por espaço (ex: 0 1 2 3):")

L = int(input("Quantidade de amostras do sinal (L): "))
N = int(input("Número de pontos da DFT (N >= L):   "))

np.random.seed(42)
x = np.random.randn(L)

X = medir_tempo_dft(x, N=N)

print(f"\n=== Saída da DFT ({N} pontos) ===")

if len(X) > 10:
    t = 10
else:
    t = len(X)
    
for i in range(t):
    print(f"  X[{i}] = {X[i]:.2f}")

=== Implementação da DFT Direta ===
Digite os valores das amostras separados por espaço (ex: 0 1 2 3):
Tamanho do sinal:  N = 1024
DFT Direta:        1894.1724 ms

=== Saída da DFT (1024 pontos) ===
  X[0] = 30.08+0.00j
  X[1] = 29.15+8.67j
  X[2] = -30.28+34.26j
  X[3] = -14.00+22.97j
  X[4] = -34.86-9.42j
  X[5] = 25.65+13.12j
  X[6] = -1.41-2.54j
  X[7] = 27.36-17.36j
  X[8] = 8.44+20.10j
  X[9] = 9.46+29.13j


In [6]:
import json

tamanhos = [16, 32, 64, 128, 256, 512, 1024, 2048]
REPETICOES = 5

tempos_ms = []

for N in tamanhos:
    np.random.seed(42)
    x = np.random.randn(N)
    medicoes = []
    for _ in range(REPETICOES):
        t0 = time.time()
        dft(x)
        medicoes.append((time.time() - t0) * 1000)
    media = round(float(np.mean(medicoes)), 4)
    tempos_ms.append(media)
    print(f"N = {N:5d} → {media:.4f} ms")

metricas = {
    "nome"       : "DFT Direta",
    "tamanhos"   : tamanhos,
    "tempos_ms"  : tempos_ms
}

with open("dft_basica_metrics.json", "w") as f:
    json.dump(metricas, f, indent=2)

print("\nSalvo em dft_basica_metrics.json")

N =    16 → 0.7924 ms
N =    32 → 2.7552 ms
N =    64 → 12.3923 ms
N =   128 → 35.3275 ms
N =   256 → 121.9202 ms
N =   512 → 510.2503 ms
N =  1024 → 2092.8924 ms
N =  2048 → 8012.3047 ms

Salvo em dft_basica_metrics.json
